# TR3D — SAW Adaptive Sweep

Sweeps the residual-quantile gate `q` over four values and three random seeds
(12 training runs) for SDF-Aware Weighting, and compares against the fixed-weight
sweep of `01_fixed_ablation.ipynb` on the same benchmark.

**What is swept, and what is not.** SAW derives the eikonal weight from
training dynamics, so there is no weight to select. The sweep instead
characterises sensitivity to `q`, the one setting that governs the gate.
`beta` is held at 0.999: four runs on the reversed vortex at 0.99 and 0.999
gave final weights within 5% of one another, so it is not a sweep dimension.

**`q = 1.0` is the ablation, not an extra experiment.** At that value the gate
is inactive and SAW reduces to plain gradient-norm balancing. It is included as
a column so the gate's contribution is measured under the same protocol as
everything else rather than reported separately.

**Two quantities are logged that the fixed sweep does not need:** the final
adaptive weight, and the iteration at which it first falls below 1e-2. Without
them the sweep records how well the method scored but not what it selected.

**Outputs:** `TR3D_saw_sweep.csv`, appended after each run so a disconnect
costs at most one run, plus a summary table with seed spread.

In [2]:
# --- setup -------------------------------------------------------------
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/TR3D'
except ImportError:
    PROJECT_DIR = os.getcwd()

sys.path.insert(0, PROJECT_DIR)
os.makedirs(PROJECT_DIR, exist_ok=True)

import csv, itertools, time
import numpy as np
import torch
import tr3d_core as core

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

Mounted at /content/drive
device: cuda


In [2]:
# --- configuration ---
Q_GRID       = [0.95, 1.00]   # 1.00 = gate off (ablation)
SEEDS        = [42, 43, 44]
BETA         = 0.999

W_PDE, W_IC  = 1.0, 10.0
N_F, N_I     = 10000, 5000
LAYERS, NEURONS = 8, 256
T_FINAL      = 5.0
ADAM_EPOCHS  = 20000
LBFGS_EPOCHS = 500
GRID_N       = 201
N_SNAPSHOTS  = 6

EST_MIN_PER_RUN = 20        # update from your first measured run
CSV_PATH = f'{PROJECT_DIR}/TR3D_saw_sweep.csv'

times, tlabels = core.snapshot_times(T_FINAL, N_SNAPSHOTS)
grid = core.Grid(n=GRID_N)
METRICS = ['rel_L2', 'vol_MAPE', 'eik_dev', 'l2_abs', 'vol_abs']

FIELDS = ['q', 'seed', 'beta']
for m in METRICS:
    FIELDS += [f'{m}_t{i}' for i in range(N_SNAPSHOTS)] + [f'avg_{m}']
FIELDS += ['w_eik_final', 'w_eik_min', 'call_below_1e-2',
           'loss_adam', 'loss_final', 'lbfgs_gain', 'minutes']

print(f'{len(Q_GRID) * len(SEEDS)} runs, {len(FIELDS)} columns per row')

6 runs, 45 columns per row


In [3]:
# --- one run -----------------------------------------------------------
def run_one(q, seed):
    """Train once at gate setting q and return a CSV row."""
    net = core.build_model(device, seed=seed, layers=LAYERS, neurons=NEURONS)
    X_f, X_i = core.sample_points(N_F, N_I, device, seed=seed)
    loss_fn = core.make_loss_saw(net, X_f, X_i, T_FINAL, beta=BETA, q=q)

    tr = core.train(net, loss_fn, adam_epochs=ADAM_EPOCHS,
                    lbfgs_epochs=LBFGS_EPOCHS, verbose=False)

    # SAW records its own weight trajectory, one entry per loss evaluation;
    # the sweep is about what the method selects, not only how well it scores
    w_hist = loss_fn.w_hist
    call_below = next((k for k, w in enumerate(w_hist) if w < 1e-2), -1)
    res = core.evaluate(net, grid, times, T_FINAL, device, cache=False)

    row = {'q': q, 'seed': seed, 'beta': BETA}
    for m in METRICS:
        for i in range(N_SNAPSHOTS):
            row[f'{m}_t{i}'] = f'{res[m][i]:.6e}'
        row[f'avg_{m}'] = f"{res['avg_' + m]:.6e}"
    row.update({
        'w_eik_final': f'{w_hist[-1]:.6e}' if w_hist else 'nan',
        'w_eik_min':   f'{min(w_hist):.6e}' if w_hist else 'nan',
        'call_below_1e-2': call_below,
        'loss_adam':  f"{tr['loss_adam']:.6e}",
        'loss_final': f"{tr['loss_final']:.6e}",
        'lbfgs_gain': f"{tr['lbfgs_gain']:.6e}",
        'minutes':    f"{tr['minutes']:.2f}"})

    np.savez_compressed(f'{PROJECT_DIR}/TR3D_saw_whist_q{q:g}_seed{seed}.npz',
                        w_hist=np.asarray(w_hist, dtype=np.float32))

    del net, X_f, X_i, loss_fn
    if device == 'cuda':
        torch.cuda.empty_cache()
    return row

In [4]:
# --- sweep (resumable) -------------------------------------------------
done = set()
if os.path.exists(CSV_PATH):
    with open(CSV_PATH) as f:
        for r in csv.DictReader(f):
            done.add((float(r['q']), int(r['seed'])))
    print(f'{len(done)} run(s) already complete — skipping those')

todo = [p for p in itertools.product(Q_GRID, SEEDS) if p not in done]
print(f'{len(todo)} run(s) to go, ~{len(todo) * EST_MIN_PER_RUN / 60:.1f} h\n',
      flush=True)

for k, (qv, sd) in enumerate(todo, 1):
    print(f'[{k}/{len(todo)}] q={qv:g} seed={sd} ...', flush=True)
    row = run_one(qv, sd)
    new = not os.path.exists(CSV_PATH) or os.path.getsize(CSV_PATH) == 0
    with open(CSV_PATH, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    print(f"    relL2={float(row['avg_rel_L2']):.4f}%  "
          f"MAPE={float(row['avg_vol_MAPE']):.4f}%  "
          f"w_eik={float(row['w_eik_final']):.3e}  "
          f"({float(row['minutes']):.1f} min)", flush=True)

6 run(s) to go, ~2.0 h

[1/6] q=0.95 seed=42 ...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:335.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    relL2=0.6086%  MAPE=2.3670%  w_eik=3.477e-02  (13.4 min)
[2/6] q=0.95 seed=43 ...
    relL2=1.7957%  MAPE=1.9623%  w_eik=3.229e-02  (13.4 min)
[3/6] q=0.95 seed=44 ...
    relL2=0.7013%  MAPE=1.1045%  w_eik=8.624e-02  (14.4 min)
[4/6] q=1 seed=42 ...
    relL2=0.2560%  MAPE=0.1668%  w_eik=1.697e-01  (13.3 min)
[5/6] q=1 seed=43 ...
    relL2=0.6416%  MAPE=0.2156%  w_eik=2.587e-01  (14.6 min)
[6/6] q=1 seed=44 ...
    relL2=nan%  MAPE=100.0000%  w_eik=nan  (18.2 min)


In [3]:
import numpy as np
w = np.load(f'{PROJECT_DIR}/TR3D_saw_whist_q1_seed44.npz')['w_hist']
bad = np.argmax(~np.isfinite(w))
print('first non-finite at call', bad, 'of', len(w))
print(w[max(0,bad-10):bad+3])

first non-finite at call 21002 of 29051
[0.81586134 0.81580013 0.81591153 0.8158309  0.8157445  0.8156578
 0.8155712  0.81548464 0.8153982  0.81531185        nan        nan
        nan]


## Summary

Unlike the fixed sweep there is no weight to select: every row is the method.
The comparison that matters is against the best row of the fixed sweep on this
benchmark, and against the `q = 1.0` column, which is SAW with the gate
disabled.

In [5]:
# --- summary -----------------------------------------------------------
rows = list(csv.DictReader(open(CSV_PATH)))
agg = {}
for r in rows:
    agg.setdefault(float(r['q']), []).append(r)

print(f"{'q':>6} | {'rel L2 (%)':>19} | {'vol MAPE (%)':>19} | "
      f"{'w_eik final':>12} | {'n':>2}")
print('-' * 72)
for qv in sorted(agg):
    g = agg[qv]
    m = {k: np.mean([float(r['avg_' + k]) for r in g]) for k in ('rel_L2', 'vol_MAPE')}
    s = {k: np.std([float(r['avg_' + k]) for r in g]) for k in ('rel_L2', 'vol_MAPE')}
    wf = np.mean([float(r['w_eik_final']) for r in g])
    tag = '  <- gate off' if qv >= 1.0 else ''
    print(f"{qv:>6.2f} | {m['rel_L2']:>9.4f} ± {s['rel_L2']:<7.4f} | "
          f"{m['vol_MAPE']:>9.4f} ± {s['vol_MAPE']:<7.4f} | "
          f"{wf:>12.3e} | {len(g):>2}{tag}")

gated = {q: v for q, v in agg.items() if q < 1.0}
if gated:
    best = min(gated, key=lambda q: np.mean([float(r['avg_rel_L2']) for r in agg[q]]))
    print(f'\nbest gated setting: q = {best:g}')
if 1.0 in agg and gated:
    on  = np.mean([float(r['avg_rel_L2']) for r in agg[best]])
    off = np.mean([float(r['avg_rel_L2']) for r in agg[1.0]])
    print(f'gate on/off: {on:.4f}% vs {off:.4f}%  (factor {off/on:.1f})')

     q |          rel L2 (%) |        vol MAPE (%) |  w_eik final |  n
------------------------------------------------------------------------
  0.95 |    1.0352 ± 0.5391  |    1.8112 ± 0.5264  |    5.110e-02 |  3
  1.00 |       nan ± nan     |   33.4608 ± 47.0503 |          nan |  3  <- gate off

best gated setting: q = 0.95
gate on/off: 1.0352% vs nan%  (factor nan)


In [6]:
# --- weight selected, against the fixed-weight optimum -----------------
# Fill in from 01_fixed_ablation.ipynb on this benchmark.
FIXED_BEST_W   = 1e-1      # weight selected by the fixed sweep
FIXED_BEST_L2  = 0.442     # its seed-averaged relative L2 (%)

for qv in sorted(agg):
    if qv >= 1.0:
        continue
    wf = np.mean([float(r['w_eik_final']) for r in agg[qv]])
    l2 = np.mean([float(r['avg_rel_L2']) for r in agg[qv]])
    print(f'q={qv:.2f}: SAW selected w_eik = {wf:.3e} '
          f'({wf / FIXED_BEST_W:.2f}x the fixed optimum), '
          f'rel L2 {l2:.4f}% vs {FIXED_BEST_L2:.4f}% fixed')

q=0.95: SAW selected w_eik = 5.110e-02 (0.51x the fixed optimum), rel L2 1.0352% vs 0.4420% fixed
